# Gene mapping
This code set genes and GPR rules in a common namespace and format, for further comparisons between models. Genes are translated into NCBI gene IDs (for A. aegypti and C. griseus) or NCBI protein ID (for E. siliculosus), and saved to complete_gene_mapping.csv. GPRs are expressed according to these consensus namespaces, transformed to Disjunctive Normal Form (with clauses sorted by alphabetic order) and saved to normalized_gprs_{org}.csv tables.

# 1. Imports

In [6]:
import gzip
import re
import pandas as pd
from typing import Dict, List, Set, Iterable
from sympy.logic.boolalg import to_dnf
from sympy.parsing.sympy_parser import parse_expr
import cobra
import time
import os

# 2. Function definition

## 2.1 Functions for ID normalization

In [131]:
def normalize_id(original_id, method, org):
    """
    Normalize IDs. In the case of AuReMe (au), Pathway Tools (pt), carveMe (cv), merlin (me), and reference E_siliculosus,
    special tokens are replaced. In the rest of cases, the original_id is returned.
    """
    replacements = [('gp_', ''),('__ZERO__', '0'), ('__ONE__', '1'),('__TWO__', '2'),('__THREE__', '3'), ('__FOUR__', '4'), 
                    ('__FIVE__', '5'),('__SIX__', '6'), ('__SEVEN__', '7'), ('__EIGHT__', '8'), ('__NINE__', '9')]
    
    if method in ('au', 'pt'):
        normalized_id = original_id
        for old, new in replacements:
            normalized_id = normalized_id.replace(old, new)
        normalized_id = normalized_id.replace('__46__','.')
        return normalized_id

    elif method in ('cv', 'me'):
        # Replace only the last underscore in each atomic token by a dot.
        def repl(match):
            token = match.group(0)
            return token[::-1].replace('_', '.', 1)[::-1]
        return re.sub(r'\b[A-Za-z0-9_]+\b', repl, original_id)

    elif method == 'ref' and org == 'E_siliculosus':
        normalized_id = original_id
        for old, new in replacements:
            normalized_id = normalized_id.replace(old, new)
        normalized_id = normalized_id.replace('__45__','-')
        return normalized_id
    
    else:
        return original_id

### Tests

In [52]:
test_id = 'gp_CBN__SEVEN____FOUR____SIX____ONE____ONE______FOUR____SIX______ONE__'
normalized_id = normalize_id(test_id, 'au', 'E_siliculosus')
print(normalized_id)

test_id = 'gp_Ec____FOUR____FIVE______ONE____FIVE_____ZERO____ZERO____FOUR____NINE____FIVE____ZERO__'
normalized_id = normalize_id(test_id, 'ref', 'E_siliculosus')
print(normalized_id)

CBN74611.1
Ec-15_004950


## 2.2 Functions for creating gene ID maps

In [1]:
def get_complete_gene_mapping(VBg_NCBIg_map, PEp_NCBIp_map, NCBIp_NCBIg_map, methods, orgs, models_dict):
    """
    Constructs a unified mapping table that, for each model, links its gene identifiers
    to normalized IDs, NCBI protein accessions, and NCBI gene IDs.
    
    Arguments:
        VBg_NCBIg_map: dict mapping VectorBase gene IDs → NCBI Gene IDs (Entrez)
        PEp_NCBIp_map: dict mapping E. siliculosus protein IDs → NCBI protein IDs
        NCBIp_NCBIg_map: dict mapping NCBI protein accessions → NCBI Gene IDs
        methods: list of reconstruction methods (e.g., ['au', 'cv', 'me', 'pt'])
        orgs: list of organism names (e.g., ['A_aegypti', 'C_griseus', 'E_siliculosus'])
        models_dict: dict mapping model IDs (e.g., 'au_A_aegypti') → COBRApy model objects

    Returns:
        A pandas DataFrame with columns: ['Original ID', 'Normalized ID', 'NCBI protein', 'NCBI gene', 'Model ID']
    """
    
    # Initialize an empty list to accumulate row dictionaries (faster than appending to a DataFrame directly)
    records = []

    # Iterate over all combinations of methods and organisms
    for method in methods:
        for org in orgs:
            model_id = f"{method}_{org}"            
            model = models_dict[model_id]
            
            for gene in model.genes:
                original_id = gene.id
                row = {"Original ID": original_id, "Model ID": model_id}

                # Normalize the gene identifier based on the reconstruction method
                normalized_id = normalize_id(original_id, method, org)
                row["Normalized ID"] = normalized_id

                # Special case 1: Reference model for E. siliculosus
                if model_id == "ref_E_siliculosus":
                    protein_id = PEp_NCBIp_map.get(normalized_id, normalized_id)
                    gene_id = NCBIp_NCBIg_map.get(protein_id, protein_id)

                # Special case 2: Reference model for A. aegypti (uses VectorBase → Entrez mapping)
                elif model_id == "ref_A_aegypti":
                    protein_id = ''
                    gene_id = VBg_NCBIg_map.get(normalized_id, normalized_id)

                # Special case 3: Reference model for C. griseus (already uses NCBI gene IDs)
                elif model_id == "ref_C_griseus":
                    protein_id = ""
                    gene_id = normalized_id

                # All other models (map normalized protein IDs → NCBI gene IDs)
                else:
                    protein_id = normalized_id 
                    gene_id = NCBIp_NCBIg_map.get(protein_id, protein_id)

                # Fill the row
                row["NCBI protein"] = protein_id
                row["NCBI gene"] = gene_id

                records.append(row)

    # Convert accumulated records into a DataFrame
    df = pd.DataFrame.from_records(records, columns=["Original ID", "Normalized ID", "NCBI protein", "NCBI gene", "Model ID"])

    # Remove '.k' suffixes from NCBI gene IDs where k is a single digit
    df["NCBI gene"] = df["NCBI gene"].astype(str).str.replace(r"\.\d$", "", regex=True)

    return df


def get_VBg_NCBIg_map(VBg_NCBIg_csv):
    """
    Return a dictionary mapping VectorBase Gene IDs to NCBI Gene IDs (Entrez).
    If a row in 'Entrez Gene ID' contains multiple comma-separated IDs,
    the numerically greatest one is selected.
    """
    df = pd.read_csv(VBg_NCBIg_csv, dtype=str)
    df = df.dropna(subset=["Gene ID", "Entrez Gene ID"])

    # Choose the numerically greatest Entrez Gene ID when multiple are present
    df["Entrez Gene ID"] = df["Entrez Gene ID"].apply(
        lambda x: (
            str(max(int(i.strip()) for i in x.split(",")))
            if pd.notnull(x)
            else np.nan
        )
    )

    return dict(zip(df["Gene ID"].astype(str), df["Entrez Gene ID"].astype(str)))
    

def get_PEp_NCBIp_map(PEp_NCBIp_csv):
    """
    Return a dictionary mapping Phaeoexplorer protein IDs to NCBI-recognizable protein IDs
    """
    try:
        df = pd.read_csv(PEp_NCBIp_csv, dtype=str, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(PEp_NCBIp_csv, dtype=str, encoding='latin1')
    
    return dict(zip(df["v2 LocusID"].astype(str), df["EMBL"].astype(str)))
    

def get_NCBIp_NCBIg_map(NCBIp_NCBIg_gz, orgs):
    """
    Build NCBI protein to gene map for some organisms (orgs) from NCBIp_NCBIg_gz.
    """

    org_to_tax_id_map = {'A_aegypti': 7159, 'C_griseus': 10029, 'E_siliculosus': 2880} # NCBI Taxon IDs 
    tax_ids = [str(org_to_tax_id_map[org]) for org in orgs]
    
    mapping: Dict[str, str] = {}
    with gzip.open(NCBIp_NCBIg_gz, "rt") as f:
        header = f.readline().rstrip("\n").split("\t")
        print(header)
        
        tax_idx = header.index("#tax_id")
        prot_idx = header.index("protein_accession.version")
        gene_idx = header.index("GeneID")

        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) <= max(tax_idx, prot_idx, gene_idx):
                continue
            if not parts[tax_idx] in tax_ids:
                continue
            prot = parts[prot_idx]
            gid = parts[gene_idx]
            if prot and gid and prot != "-":
                mapping[prot] = gid
    return mapping

## 2.3 Functions for parsing GPR into clauses

In [133]:
_TOKEN_RE = re.compile(r"\(|\)|\band\b|\bor\b|[A-Za-z0-9_.:-]+")

def _tokenize_gpr(gpr: str) -> List[str]:
    # Canonicalize operator case and tokenize
    gpr = gpr.replace("AND", "and").replace("OR", "or")
    gpr = gpr.replace(" And ", " and ").replace(" Or ", " or ")
    return [t.group(0) for t in _TOKEN_RE.finditer(gpr)]


def to_dnf_string(gpr: str) -> str:
    """
    Convert a GPR string to DNF using Sympy, preserving original atomic tokens via
    a temporary bijection (handles dots/colons/underscores safely).
    Returns a string like: "(a and b) or (c)" (not yet translated).
    """
    if not gpr or not gpr.strip():
        return ""

    tokens = _tokenize_gpr(gpr)
    atoms: List[str] = [t for t in tokens if t not in ("(", ")", "and", "or")]
    uniq_atoms = []
    seen = set()
    for a in atoms:
        if a not in seen:
            uniq_atoms.append(a)
            seen.add(a)

    # Build a safe variable map: atom -> X0, X1, ...
    forward = {atom: f"X{i}" for i, atom in enumerate(uniq_atoms)}
    backward = {v: k for k, v in forward.items()}

    # Build an expression string for sympy
    expr_parts = []
    for t in tokens:
        if t == "and":
            expr_parts.append("&")
        elif t == "or":
            expr_parts.append("|")
        elif t in ("(", ")"):
            expr_parts.append(t)
        else:
            expr_parts.append(forward[t])
    expr_str = " ".join(expr_parts)

    try:
        expr = parse_expr(expr_str)
        dnf_expr = to_dnf(expr, simplify=True)
        dnf_str = str(dnf_expr)
    except Exception:
        # If parsing fails, return the original GPR as-is
        return gpr

    # Map back to original atom strings and normalize operators/spaces
    # dnf_str uses '&' and '|' and variable names like X0
    def restore_atoms(s: str) -> str:
        # Replace variable names with original atoms (use longest names first just in case)
        for var, atom in sorted(backward.items(), key=lambda kv: -len(kv[0])):
            s = re.sub(rf"\b{re.escape(var)}\b", atom, s)
        s = s.replace("&", "and").replace("|", "or")
        # Add spaces around operators for consistent splitting later
        s = re.sub(r'\s*and\s*', ' and ', s)
        s = re.sub(r'\s*or\s*', ' or ', s)
        s = re.sub(r'\s+', ' ', s).strip()
        return s

    return restore_atoms(dnf_str)


def split_dnf_into_clauses(dnf_gpr: str) -> List[str]:
    """
    Split a DNF string "(a and b) or (c)" into a list of clause strings:
    ["a and b", "c"]
    """
    if not dnf_gpr:
        return []
    # Split on top-level ' or ' (DNF ensures top-level ORs)
    parts = [p.strip() for p in dnf_gpr.split(" or ")]
    clauses = []
    for p in parts:
        # remove outer parentheses if present
        if p.startswith("(") and p.endswith(")"):
            p = p[1:-1].strip()
        clauses.append(p)
    return [c for c in clauses if c]

### Tests

In [58]:
test_gpr = '(412551 and (004411 or 514141)) or (454132 and 515151)'
print(f"Original GPR: {test_gpr}\n")

test_gpr_list = _tokenize_gpr(test_gpr)
print(f"Original GPR as list: ")
print(test_gpr_list)
print(f"\n")

test_gpr_dnf = to_dnf_string(test_gpr)
print(f"GPR in Disjunctive Normal Form: {test_gpr_dnf}\n")

Original GPR: (412551 and (004411 or 514141)) or (454132 and 515151)

Original GPR as list: 
['(', '412551', 'and', '(', '004411', 'or', '514141', ')', ')', 'or', '(', '454132', 'and', '515151', ')']


GPR in Disjunctive Normal Form: (412551 and 004411) or (412551 and 514141) or (454132 and 515151)



## 2.4 Functions for translating GPR clauses

In [134]:
def translate_gpr_clauses(clause_set, complete_gene_mapping):
    """
    Translate a set of DNF clauses into NCBI GeneID, using a complete_gene_mapping. Returns a set of translated clauses.
    """
    
    complete_gene_dict = dict(zip(complete_gene_mapping["Original ID"].astype(str), complete_gene_mapping["NCBI gene"].astype(str)))
    translated: Set[str] = set()

    for clause in clause_set:
        atoms = [a.strip() for a in clause.split("and")]
        gene_ids = [complete_gene_dict[a] for a in atoms if a in complete_gene_dict]
        if gene_ids:
            translated.add(" and ".join(gene_ids))
    return translated

### Tests

In [61]:
TAX_IDS = {'A_aegypti': 7159, 'C_griseus': 10029, 'E_siliculosus': 2880} # NCBI Taxon IDs 

gpr = 'XP_001653737_2 or (XP_001656430_1 and XP_001657687_1) or XP_001661213_2 or (XP_021711953_1 or XP_021712943_1)'
print(f"Original gpr: {gpr}\n")

dnf_gpr = to_dnf_string(gpr)
print(f"DNF gpr: {dnf_gpr}")

clause_set = split_dnf_into_clauses(dnf_gpr)
print("Clauses are: ")
print(clause_set)

translated_gpr_clauses = translate_gpr_clauses(clause_set, complete_gene_mapping)
print("Translated clauses are: ")
print(translated_gpr_clauses)

Original gpr: XP_001653737_2 or (XP_001656430_1 and XP_001657687_1) or XP_001661213_2 or (XP_021711953_1 or XP_021712943_1)

DNF gpr: XP_001653737_2 or XP_001661213_2 or XP_021711953_1 or XP_021712943_1 or (XP_001656430_1 and XP_001657687_1)
Clauses are: 
['XP_001653737_2', 'XP_001661213_2', 'XP_021711953_1', 'XP_021712943_1', 'XP_001656430_1 and XP_001657687_1']
Translated clauses are: 
{'5567085', '5577354 and 5567804', '5574147', '5571642', '5577668'}


## 2.5 Functions for generating normalized GPR mapping tables

In [135]:
def generate_normalized_gpr_tables(models_dict, complete_rxn_mapping, organisms, methods, complete_gene_mapping):
    """
    Generates a normalized GPR table for each organism. Each table has one row per MNXR_id and columns for each method: 
    ['MNXR_id', 'GPR_ref', 'GPR_method2', ..., 'rxns_ref', 'rxns_method2', ...]. Includes robust matching for reaction IDs 
    by trying different normalizations (remove 'R_' prefix, replace '_' by '-', etc.).
    """

    rxn_map = pd.read_csv(complete_rxn_mapping, dtype=str).fillna("")

    # Only MNXR targets
    mnxr_ids: set[str] = {rid for rid in rxn_map["Final ID"] if isinstance(rid, str) and rid.startswith("MNXR")}
    organism_tables = {}

    # --- Helper to robustly get a reaction object ---
    def get_reaction_safe(model, rxn_id):
        """
        Try multiple ID variants to find a matching reaction in the model.
        """
        
        tried = [rxn_id]

        # Generate variants
        variants = set()
        if rxn_id.startswith("R_"):
            variants.add(rxn_id[2:])
        if "_" in rxn_id:
            variants.add(rxn_id.replace("_", "-"))

        for var in variants:
            tried.append(var)
            if var in model.reactions:
                return model.reactions.get_by_id(var), tried

        # If still not found, return None and the tried list for debugging
        if rxn_id in model.reactions:
            return model.reactions.get_by_id(rxn_id), tried

        return None, tried

    for org in organisms:
        rows = []
        mnxr_ids_for_org = set(rxn_map.loc[rxn_map["model"].str.endswith(f"_{org}"), "Final ID"].dropna().unique())
        mnxr_ids_for_org = [mnx for mnx in mnxr_ids_for_org if mnx.startswith("MNXR")]

        for mnxr in mnxr_ids_for_org:
            row = {"MNXR_id": mnxr}

            for mth in methods:
                model_id = f"{mth}_{org}"

                if model_id not in models_dict:
                    row[f"GPR_{mth}"] = "N/A"
                    row[f"rxns_{mth}"] = "N/A"
                    continue

                model = models_dict[model_id]
                subset = rxn_map[(rxn_map["Final ID"] == mnxr) & (rxn_map["model"] == model_id)]
                original_ids = subset["Original ID"].tolist()

                if not original_ids:
                    row[f"rxns_{mth}"] = "N/A"
                    row[f"GPR_{mth}"] = "N/A"
                    continue

                row[f"rxns_{mth}"] = ", ".join(original_ids)
                joint_clause_set = set()

                for rxn_id in original_ids:
                    rxn, tried = get_reaction_safe(model, rxn_id)
                    if rxn is None:
                        # Optional: debugging line if needed
                        # print(f" {rxn_id} not found in {model_id}. Tried: {tried}")
                        continue

                    gpr = (rxn.gene_reaction_rule or "").strip()
                    if not gpr:
                        continue

                    dnf_gpr = to_dnf_string(gpr)
                    for clause in split_dnf_into_clauses(dnf_gpr):
                        if clause:
                            joint_clause_set.add(clause)

                if not joint_clause_set:
                    row[f"GPR_{mth}"] = "N/A"
                    continue

                translated_set = translate_gpr_clauses(joint_clause_set, complete_gene_mapping)
                if not translated_set:
                    row[f"GPR_{mth}"] = "N/A"
                    continue

                # Sort atoms within each clause
                sorted_clauses = []
                for cl in translated_set:
                    atoms = [a.strip() for a in cl.split("and") if a.strip()]
                    atoms_sorted = sorted(atoms)
                    sorted_clauses.append(" and ".join(atoms_sorted))
                joint_gpr_list = sorted(sorted_clauses)
                joint_gpr_str = " or ".join([f"({c})" if " and " in c else c for c in joint_gpr_list])
                row[f"GPR_{mth}"] = joint_gpr_str

            rows.append(row)

        gpr_cols = [f"GPR_{m}" for m in methods]
        rxn_cols = [f"rxns_{m}" for m in methods]
        columns = ["MNXR_id"] + gpr_cols + rxn_cols
        df_org = pd.DataFrame(rows, columns=columns)
        organism_tables[org] = df_org

    return organism_tables

In [145]:
# === Settings ===
org = "A_aegypti"
method = "me"
model_id = f"{method}_{org}"
rxn_map_path = "mappings/complete_rxn_mapping.csv"
normalized_path = f"mappings/normalized_gprs_{org}.csv"
complete_gene_mapping = pd.read_csv('mappings/complete_gene_mapping.csv', dtype=str).fillna("") #Import complete gene mapping
rxn_map = pd.read_csv(rxn_map_path, dtype=str).fillna("")
normalized_gprs = pd.read_csv(normalized_path)
model = models_dict[model_id]

# === Auxiliary functions ===
def get_prefix(rid: str):
    """
    Return the prefix before the first '_'.
    """
    
    return rid.split("_")[0] if isinstance(rid, str) else None

def find_reaction_by_prefix(model, rxn_id):
    """
    Finds a reaction sharing prefix with rxn_id.
    """
    
    prefix = get_prefix(rxn_id)
    if not prefix:
        return None
    for r in model.reactions:
        if get_prefix(r.id) == prefix:
            return r
    return None

# === Processing ===
mnxr_ids_for_org = (rxn_map.loc[rxn_map["model"] == model_id, "Final ID"].dropna().unique().tolist())
mnxr_ids_for_org = [m for m in mnxr_ids_for_org if m.startswith("MNXR")]
updated_gprs = normalized_gprs.copy()

for mnxr in mnxr_ids_for_org:
    subset = rxn_map[(rxn_map["Final ID"] == mnxr) & (rxn_map["model"] == model_id)]
    original_ids = subset["Original ID"].tolist()

    joint_clause_set = set()

    for rxn_id in original_ids:
        rxn = find_reaction_by_prefix(model, rxn_id)
        if rxn is None:
            continue

        gpr = (rxn.gene_reaction_rule or "").strip()
        if not gpr:
            continue

        dnf_gpr = to_dnf_string(gpr)
        for clause in split_dnf_into_clauses(dnf_gpr):
            if clause:
                joint_clause_set.add(clause)

    if not joint_clause_set:
        new_gpr_value = "N/A"
    else:
        translated_set = translate_gpr_clauses(joint_clause_set, complete_gene_mapping)
        if not translated_set:
            new_gpr_value = "N/A"
        else:
            sorted_clauses = []
            for cl in translated_set:
                atoms = [a.strip() for a in cl.split("and") if a.strip()]
                atoms_sorted = sorted(atoms)
                sorted_clauses.append(" and ".join(atoms_sorted))
            joint_gpr_list = sorted(sorted_clauses)
            new_gpr_value = " or ".join([f"({c})" if " and " in c else c for c in joint_gpr_list])

    updated_gprs.loc[updated_gprs["MNXR_id"] == mnxr, f"GPR_{method}"] = new_gpr_value

# === Report ===
num_updated = (updated_gprs[f"GPR_{method}"] != normalized_gprs[f"GPR_{method}"]).sum()
print(f"✅ {num_updated} GPR_me entries updated for {org} using prefix matching only.")

# === (Optional) see first changes ===
changed = updated_gprs.loc[updated_gprs[f"GPR_{method}"] != normalized_gprs[f"GPR_{method}"], ["MNXR_id", f"GPR_{method}"]]
print(changed.head(10))


# === Save results ===
updated_gprs = updated_gprs.fillna("N/A")
output_path = "mappings/normalized_gprs_A_aegypti(corrected_GPR_me).csv"
updated_gprs.to_csv(output_path, index=False)
print(f"File saved at: {output_path}")

C:\Users\futbo\AppData\Local\Temp\ipykernel_35144\2574121357.py:78: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '5569494 or 5577499' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  updated_gprs.loc[


✅ 9885 GPR_me entries updated for A_aegypti using prefix matching only.
      MNXR_id                                             GPR_me
0  MNXR100242                                                NaN
1  MNXR157322                                                NaN
2  MNXR158723                                                NaN
3  MNXR164434                                                NaN
4  MNXR211248                                                NaN
5  MNXR146698  110676362 or 110676363 or 110676364 or 1106769...
6  MNXR119161                                                NaN
7  MNXR190906                                                NaN
8  MNXR106983                                                NaN
9  MNXR146701  110676362 or 110676363 or 110676364 or 1106769...
Archivo guardado en: mappings/normalized_gprs_A_aegypti(corrected_GPR_me).csv


# 3. Create gene mapping

## 3.1 Create partial maps

In [35]:
VBg_NCBIg_csv = 'mappings/gene_mapping_sources/VBg_NCBIg.csv'
VBg_NCBIg_map = get_VBg_NCBIg_map(VBg_NCBIg_csv)

PEp_NCBIp_csv = 'mappings/gene_mapping_sources/PEp_NCBIp.csv'
PEp_NCBIp_map = get_PEp_NCBIp_map(PEp_NCBIp_csv)

orgs = ['A_aegypti', 'C_griseus', 'E_siliculosus']
NCBIp_NCBIg_gz = 'mappings/gene_mapping_sources/NCBIp_NCBIg.gz'
NCBIp_NCBIg_map = get_NCBIp_NCBIg_map(NCBIp_NCBIg_gz, orgs)

['#tax_id', 'GeneID', 'status', 'RNA_nucleotide_accession.version', 'RNA_nucleotide_gi', 'protein_accession.version', 'protein_gi', 'genomic_nucleotide_accession.version', 'genomic_nucleotide_gi', 'start_position_on_the_genomic_accession', 'end_position_on_the_genomic_accession', 'orientation', 'assembly', 'mature_peptide_accession.version', 'mature_peptide_gi', 'Symbol']


## 3.2 Create model dict

In [36]:
input_model_path = 'models' # Folder with generated models
models_dict = dict()

#A. aegypti models
ref_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'reference/A_aegypti.xml') #AuReMe
au_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'AuReMe/A_aegypti.xml') #AuReMe
cv_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'carveMe/A_aegypti.xml') #carveMe
me_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'merlin/A_aegypti.xml') #merlin
ms_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'modelseed/A_aegypti.xml') #modelSEED
pt_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'pathway_tools/A_aegypti.xml') #Pathway tools
rm_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_metacyc/A_aegypti.xml') #Raven metacyc
rk_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_kegg/A_aegypti.xml') #Raven kegg
rc_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_comb/A_aegypti.xml') #Raven combined
rec_A_aegypti = cobra.io.read_sbml_model(input_model_path + '/' + 'reconstructor/A_aegypti.sbml') #Reconstructor

models_dict['ref_A_aegypti'] = ref_A_aegypti
models_dict['au_A_aegypti'] = au_A_aegypti
models_dict['cv_A_aegypti'] = cv_A_aegypti
models_dict['me_A_aegypti'] = me_A_aegypti
models_dict['ms_A_aegypti'] = ms_A_aegypti
models_dict['pt_A_aegypti'] = pt_A_aegypti
models_dict['rm_A_aegypti'] = rm_A_aegypti
models_dict['rk_A_aegypti'] = rk_A_aegypti
models_dict['rc_A_aegypti'] = rc_A_aegypti
models_dict['rec_A_aegypti'] = rec_A_aegypti

#E. siliculosus models
ref_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'reference/E_siliculosus.xml')
au_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'AuReMe/E_siliculosus.xml') #AuReMe
cv_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'carveMe/E_siliculosus.xml') #carveMe
me_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'merlin/E_siliculosus.xml') #merlin
ms_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'modelseed/E_siliculosus.xml') #modelSEED
pt_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'pathway_tools/E_siliculosus.xml') #Pathway Tools
rm_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_metacyc/E_siliculosus.xml') #Raven metacyc
rk_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_kegg/E_siliculosus.xml') #Raven kegg
rc_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_comb/E_siliculosus.xml') #Raven combined
rec_E_siliculosus = cobra.io.read_sbml_model(input_model_path + '/' + 'reconstructor/E_siliculosus.sbml') #Reconstructor

models_dict['ref_E_siliculosus'] = ref_E_siliculosus
models_dict['au_E_siliculosus'] = au_E_siliculosus
models_dict['cv_E_siliculosus'] = cv_E_siliculosus
models_dict['me_E_siliculosus'] = me_E_siliculosus
models_dict['ms_E_siliculosus'] = ms_E_siliculosus
models_dict['pt_E_siliculosus'] = pt_E_siliculosus
models_dict['rm_E_siliculosus'] = rm_E_siliculosus
models_dict['rk_E_siliculosus'] = rk_E_siliculosus
models_dict['rc_E_siliculosus'] = rc_E_siliculosus
models_dict['rec_E_siliculosus'] = rec_E_siliculosus

#C. griseus models
ref_C_griseus = cobra.io.read_sbml_model(input_model_path + '/' + 'reference/C_griseus.xml')
au_C_griseus = cobra.io.read_sbml_model(input_model_path + '/' + 'AuReMe/C_griseus.xml') #AuReMe
cv_C_griseus = cobra.io.read_sbml_model(input_model_path + '/' + 'carveMe/C_griseus.xml') #carveMe
me_C_griseus = cobra.io.read_sbml_model(input_model_path + '/' + 'merlin/C_griseus.xml') #merlin
ms_C_griseus = cobra.io.read_sbml_model(input_model_path + '/' + 'modelseed/C_griseus.xml') #modelSEED
pt_C_griseus = cobra.io.read_sbml_model(input_model_path + '/' + 'pathway_tools/C_griseus.xml') #Pathway tools
rm_C_griseus = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_metacyc/C_griseus.xml') #Raven metacyc
rk_C_griseus = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_kegg/C_griseus.xml') #Raven kegg
rc_C_griseus = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_comb/C_griseus.xml') #Raven combined
rec_C_griseus = cobra.io.read_sbml_model(input_model_path + '/' + 'reconstructor/C_griseus.sbml') #Reconstructor

models_dict['ref_C_griseus'] = ref_C_griseus
models_dict['au_C_griseus'] = au_C_griseus
models_dict['cv_C_griseus'] = cv_C_griseus
models_dict['me_C_griseus'] = me_C_griseus
models_dict['ms_C_griseus'] = ms_C_griseus
models_dict['pt_C_griseus'] = pt_C_griseus
models_dict['rm_C_griseus'] = rm_C_griseus
models_dict['rk_C_griseus'] = rk_C_griseus
models_dict['rc_C_griseus'] = rc_C_griseus
models_dict['rec_C_griseus'] = rec_C_griseus

models_dict = {
    'ref_A_aegypti' : ref_A_aegypti,
    'au_A_aegypti' : au_A_aegypti,
    'cv_A_aegypti' : cv_A_aegypti,
    'me_A_aegypti' : me_A_aegypti,
    'ms_A_aegypti' : ms_A_aegypti,
    'pt_A_aegypti' : pt_A_aegypti,
    'rm_A_aegypti' : rm_A_aegypti,
    'rk_A_aegypti' : rk_A_aegypti,
    'rc_A_aegypti' : rc_A_aegypti,
    'rec_A_aegypti' : rec_A_aegypti,
    'ref_C_griseus' : ref_C_griseus,
    'au_C_griseus' : au_C_griseus,
    'cv_C_griseus' : cv_C_griseus,
    'me_C_griseus' : me_C_griseus,
    'ms_C_griseus' : ms_C_griseus,
    'pt_C_griseus' : pt_C_griseus,
    'rm_C_griseus' : rm_C_griseus,
    'rk_C_griseus' : rk_C_griseus,
    'rc_C_griseus' : rc_C_griseus,
    'rec_C_griseus' : rec_C_griseus,
    'ref_E_siliculosus' : ref_E_siliculosus,
    'au_E_siliculosus' : au_E_siliculosus,
    'cv_E_siliculosus' : cv_E_siliculosus,
    'me_E_siliculosus' : me_E_siliculosus,
    'ms_E_siliculosus' : ms_E_siliculosus,
    'pt_E_siliculosus' : pt_E_siliculosus,
    'rm_E_siliculosus' : rm_E_siliculosus,
    'rk_E_siliculosus' : rk_E_siliculosus,
    'rc_E_siliculosus' : rc_E_siliculosus,
    'rec_E_siliculosus' : rec_E_siliculosus}

Loading SBML model without fbc:strict="true"
Loading SBML with fbc-v1 (models should be encoded using fbc-v2)
No objective in listOfObjectives
No objective coefficients in model. Unclear what should be optimized
7159 does not conform to 'http(s)://identifiers.org/collection/id' or'http(s)://identifiers.org/COLLECTION:id
Model does not contain SBML fbc package information.
SBML package 'layout' not supported by cobrapy, information is not parsed
SBML package 'render' not supported by cobrapy, information is not parsed
Use of the species charge attribute is discouraged, use fbc:charge instead: <Species M_cpd00001_c0 "H2O_c0">
Use of the species charge attribute is discouraged, use fbc:charge instead: <Species M_cpd00009_c0 "Phosphate_c0">
Use of the species charge attribute is discouraged, use fbc:charge instead: <Species M_cpd00012_c0 "PPi_c0">
Use of the species charge attribute is discouraged, use fbc:charge instead: <Species M_cpd00067_c0 "H_plus__c0">
Use of the species charge attri

## 3.3 Create complete gene mapping

In [56]:
#orgs = ['A_aegypti', 'C_griseus', 'E_siliculosus']
#methods = ['ref','au','cv','me','ms','pt','rm','rk','rc','rec']
#complete_gene_mapping = get_complete_gene_mapping(VBg_NCBIg_map, PEp_NCBIp_map, NCBIp_NCBIg_map, methods, orgs, models_dict)

['#tax_id', 'GeneID', 'status', 'RNA_nucleotide_accession.version', 'RNA_nucleotide_gi', 'protein_accession.version', 'protein_gi', 'genomic_nucleotide_accession.version', 'genomic_nucleotide_gi', 'start_position_on_the_genomic_accession', 'end_position_on_the_genomic_accession', 'orientation', 'assembly', 'mature_peptide_accession.version', 'mature_peptide_gi', 'Symbol']


## 3.4 Save complete gene mapping

In [49]:
#complete_gene_mapping.to_csv("mappings/complete_gene_mapping.csv", index=False)

# 4. Map GPR rules

## 4.1 Generate the reaction to GPR mapping table

In [136]:
complete_gene_mapping = pd.read_csv('mappings/complete_gene_mapping.csv', dtype=str).fillna("") #Import complete gene mapping

complete_rxn_mapping = 'mappings/complete_rxn_mapping.csv'
organisms = ['A_aegypti', 'C_griseus', 'E_siliculosus']
methods = ['ref','au','cv','me','ms','pt','rm','rk','rc','rec']

# --- Run the main function ---
df = generate_normalized_gpr_tables(models_dict, complete_rxn_mapping, organisms, methods, complete_gene_mapping)

# --- Save the results ---
df['A_aegypti'].to_csv('mappings/normalized_gprs_A_aegypti.csv', index=False)
df['C_griseus'].to_csv('mappings/normalized_gprs_C_griseus.csv', index=False)
df['E_siliculosus'].to_csv('mappings/normalized_gprs_E_siliculosus.csv', index=False)

print("✅ Normalized GPR tables saved")

✅ Normalized GPR tables saved


# (Extra) Compartment mapping

In [84]:
rows = []

for model_id in models_dict.keys():
    model = models_dict[model_id]
    compartment_dict = model.compartments

    for compartment_id, compartment_name in compartment_dict.items():
        rows.append({
            'Original ID': compartment_id,
            'Original name': compartment_name,
            'Model': model_id,
            'Final name': '',
            'Final ID': ''
        })

# Crear el DataFrame una sola vez
df = pd.DataFrame(rows, columns=['Original ID', 'Original name', 'Model', 'Final name', 'Final ID'])

# Exportar a CSV
df.to_csv('mappings/complete_compartment_mapping.csv', index=False)

In [129]:
print(models_dict)

{'ref_A_aegypti': <Model A_aegypti at 0x1c50c4fdcd0>, 'au_A_aegypti': <Model draft at 0x1c5424d6c90>, 'cv_A_aegypti': <Model GCF_002204515_2_AaegL5_0_protein at 0x1c54d8453d0>, 'me_A_aegypti': <Model model_aaegypti at 0x1c550e37e90>, 'ms_A_aegypti': <Model AedesFastafaa at 0x1c5796d3ad0>, 'pt_A_aegypti': <Model A_aegypti at 0x1c5796d3b90>, 'rm_A_aegypti': <Model AedesRaven at 0x1c58ad21190>, 'rk_A_aegypti': <Model aedesKEGGHMMs at 0x1c591f1c350>, 'rc_A_aegypti': <Model COMBINED at 0x1c591f1c410>, 'rec_A_aegypti': <Model new_model at 0x1c59a0a6ed0>, 'ref_C_griseus': <Model iCHOv1 at 0x1c235c1a510>, 'au_C_griseus': <Model draft at 0x1c1533c3f50>, 'cv_C_griseus': <Model GCA_000223135_1_CriGri_1_0_protein at 0x1c41ca459d0>, 'me_C_griseus': <Model model_cgriseus at 0x1c3e1a64410>, 'ms_C_griseus': <Model CHOtestFastaAA at 0x1c40b922450>, 'pt_C_griseus': <Model CHO at 0x1c40b922810>, 'rm_C_griseus': <Model ChoRaven at 0x1c423c70a10>, 'rk_C_griseus': <Model ChoKEGGHMMs at 0x1c4235422d0>, 'rc_C

## (Extra) Debugging GPR mapping

In [103]:
gpr_example = 'CBN77946_1'
dnf_str = to_dnf_string(gpr_example)
clauses = split_dnf_into_clauses(dnf_str)

complete_gene_mapping = pd.read_csv('mappings/complete_gene_mapping.csv')
translated = translate_gpr_clauses(clauses, complete_gene_mapping)
print("DNF:", dnf_str)
print("Clauses:", clauses)
print("Translated:", translated)

DNF: CBN77946_1
Clauses: ['CBN77946_1']
Translated: {'CBN77946.1'}


In [98]:
gpr_example = 'gp_XP___ZERO____ZERO____ONE____SIX____FIVE____NINE____NINE____THREE____SEVEN______FOUR____SIX______TWO__ or gp_XP___ZERO____ZERO____ONE____SIX____FIVE____NINE____NINE____THREE____SIX______FOUR____SIX______TWO__ or gp_XP___ZERO____ZERO____ONE____SIX____FIVE____NINE____NINE____THREE____EIGHT______FOUR____SIX______TWO__'
dnf_str = to_dnf_string(gpr_example)
clauses = split_dnf_into_clauses(dnf_str)

complete_gene_mapping = pd.read_csv('mappings/complete_gene_mapping.csv')
translated = translate_gpr_clauses(clauses, complete_gene_mapping)
print("DNF:", dnf_str)
print("Clauses:", clauses)
print("Translated:", translated)

DNF: gp_XP___ZERO____ZERO____ONE____SIX____FIVE____NINE____NINE____THREE____SEVEN______FOUR____SIX______TWO__ or gp_XP___ZERO____ZERO____ONE____SIX____FIVE____NINE____NINE____THREE____SIX______FOUR____SIX______TWO__ or gp_XP___ZERO____ZERO____ONE____SIX____FIVE____NINE____NINE____THREE____EIGHT______FOUR____SIX______TWO__
Clauses: ['gp_XP___ZERO____ZERO____ONE____SIX____FIVE____NINE____NINE____THREE____SEVEN______FOUR____SIX______TWO__', 'gp_XP___ZERO____ZERO____ONE____SIX____FIVE____NINE____NINE____THREE____SIX______FOUR____SIX______TWO__', 'gp_XP___ZERO____ZERO____ONE____SIX____FIVE____NINE____NINE____THREE____EIGHT______FOUR____SIX______TWO__']
Translated: {'5571801', '5571804'}


In [112]:
input_model_path = 'models' # Folder with generated models
m = cobra.io.read_sbml_model(input_model_path + '/' + 'raven_comb/A_aegypti.xml')


#gprs = [r.gene_reaction_rule for r in m.reactions if r.gene_reaction_rule]
#print(len(gprs), "reacciones con GPRs")
#print("Ejemplo:", gprs[:5])


model_id = "rc_A_aegypti" 
rxn_map = pd.read_csv('mappings/complete_rxn_mapping.csv', dtype=str).fillna("")
rxn_ids_in_mapping = set(rxn_map.loc[rxn_map["model"] == model_id, "Original ID"])
rxn_ids_in_model = set(r.id for r in m.reactions)

intersection = len(rxn_ids_in_mapping & rxn_ids_in_model)
print(f"{model_id}: {intersection}/{len(rxn_ids_in_mapping)} Original IDs están en el modelo")


rc_A_aegypti: 2572/2573 Original IDs están en el modelo


In [115]:
rxn_map = pd.read_csv('mappings/complete_rxn_mapping.csv', dtype=str).fillna("")

#for rxn in m.reactions: print(rxn.id)
for model_id in models_dict.keys():
    m = models_dict[model_id]
    rxn_ids_in_mapping = set(rxn_map.loc[rxn_map["model"] == model_id, "Original ID"])
    rxn_ids_in_model = set(r.id for r in m.reactions)
    intersection = len(rxn_ids_in_mapping & rxn_ids_in_model)
    print(f"{model_id}: {intersection}/{len(rxn_ids_in_mapping)} Original IDs están en el modelo")

ref_A_aegypti: 5625/5678 Original IDs están en el modelo
au_A_aegypti: 29/1851 Original IDs están en el modelo
cv_A_aegypti: 1079/2173 Original IDs están en el modelo
me_A_aegypti: 794/2913 Original IDs están en el modelo
ms_A_aegypti: 1121/1673 Original IDs están en el modelo
pt_A_aegypti: 0/3058 Original IDs están en el modelo
rm_A_aegypti: 0/1365 Original IDs están en el modelo
rk_A_aegypti: 1723/1723 Original IDs están en el modelo
rc_A_aegypti: 2572/2573 Original IDs están en el modelo
rec_A_aegypti: 1372/1373 Original IDs están en el modelo
ref_C_griseus: 6608/6663 Original IDs están en el modelo
au_C_griseus: 2589/3484 Original IDs están en el modelo
cv_C_griseus: 1155/2325 Original IDs están en el modelo
me_C_griseus: 3318/3324 Original IDs están en el modelo
ms_C_griseus: 1121/1673 Original IDs están en el modelo
pt_C_griseus: 0/3722 Original IDs están en el modelo
rm_C_griseus: 0/878 Original IDs están en el modelo
rk_C_griseus: 1105/1105 Original IDs están en el modelo
rc_C_

In [116]:
org_name = "A_aegypti"
method_abb = "pt"
model_id = f"{method_abb}_{org_name}"

# 1. Cargar el modelo actual
model = models_dict[model_id]

# 3. Extraer los IDs originales que el mapeo tiene para este modelo
mapped_ids = rxn_map.loc[rxn_map["model"] == model_id, "Original ID"]

# 4. Extraer los IDs actuales del modelo
model_ids = [r.id for r in model.reactions]

# 5. Revisar los primeros 10 que no coinciden
diff = set(mapped_ids) - set(model_ids)
print("Ejemplo de Original IDs no presentes en modelo:")
print(list(diff)[:20])

# 6. Revisar diferencias comunes en caracteres
import re
def normalize(s):
    return re.sub(r'[^A-Za-z0-9]', '', s).lower()

norm_diff = {normalize(x) for x in diff}
norm_model = {normalize(x) for x in model_ids}
intersection_norm = len(norm_diff & norm_model)
print(f"Coincidencias tras normalizar: {intersection_norm} de {len(mapped_ids)}")

Ejemplo de Original IDs no presentes en modelo:
['ALKAPHOSPHA_RXN', 'RXN_23469', 'RXN_9024', 'ACID_PHOSPHATASE_RXN', 'GLUTARYL_COA_DEHYDROGENASE_RXN', 'RXN_17729', 'RXN_13224', 'FORMYLTETRAHYDROFOLATE_DEHYDROGENASE_RXN', 'R17_RXN', 'SAMDECARB_RXN', 'RXN0_7230_CHOLINE/2_METHYL_3_PHYTYL_14_NAPHTHOQUINONE//BETAINE_ALDEHYDE/CPD_12831.72.', '4_HYDROXYPHENYLPYRUVATE_DIOXYGENASE_RXN', 'RXN66_14', 'RXN_13684_CPD_26168/WATER//S_METHYL_L_CYSTEINE/ACET.42.', 'RXN_17127', 'RXN_12518', '3.4.14.10_RXN', 'RXN_23473', 'TREHALOSEPHOSPHA_RXN', '1.14.11.2_RXN']
Coincidencias tras normalizar: 2827 de 3159


In [119]:
# Cargar el mapping completo
mapping = pd.read_csv("mappings/complete_rxn_mapping.csv")

# Asegurarse de que la columna "Original ID" esté en formato string
mapping["Original ID"] = mapping["Original ID"].astype(str)

# Diccionario donde guardaremos los resultados
id_diff_summary = {}

for model_key, model in models_dict.items():
    # IDs presentes en el modelo
    model_rxn_ids = set([rxn.id for rxn in model.reactions])
    
    # Filtrar filas correspondientes a este modelo
    subset = mapping[mapping["model"] == model_key] if "model" in mapping.columns else mapping
    
    # IDs de mapping
    mapped_ids = set(subset["Original ID"].dropna().astype(str))
    
    # Identificar diferencias
    missing_in_model = mapped_ids - model_rxn_ids
    new_in_model = model_rxn_ids - mapped_ids
    
    id_diff_summary[model_key] = {
        "mapped_total": len(mapped_ids),
        "model_total": len(model_rxn_ids),
        "missing_in_model": len(missing_in_model),
        "new_in_model": len(new_in_model),
        "missing_examples": list(missing_in_model)[:10],
        "new_examples": list(new_in_model)[:10],
    }

# Mostrar un resumen compacto
for key, diff in id_diff_summary.items():
    print(f"\n=== {key} ===")
    print(f"Mapped: {diff['mapped_total']}, Model: {diff['model_total']}")
    print(f"Missing in model: {diff['missing_in_model']}, New in model: {diff['new_in_model']}")
    print(f"Example missing IDs: {diff['missing_examples']}")
    print(f"Example new IDs: {diff['new_examples']}")


=== ref_A_aegypti ===
Mapped: 5678, Model: 5678
Missing in model: 53, New in model: 53
Example missing IDs: ['EX_aicarm_LPAREN_e_RPAREN_', 'EX_bildglcurm_LPAREN_e_RPAREN_', 'EX_thrm_L_LPAREN_e_RPAREN_', 'H3MTerm_L', 'DM_gpi_sig_erm_', 'EX_form_LPAREN_e_RPAREN_', 'H8MTerm_U', 'EX_tyrm_L_LPAREN_e_RPAREN_', 'EX_maltttrm_LPAREN_e_RPAREN_', 'Serm_Thrtg']
Example new IDs: ['DM_dgpi_prot_hs_r_', 'PCHOLPr_hs', 'EX_nrpphr_LPAREN_e_RPAREN_', 'DM_pe_hs_LPAREN_r_RPAREN_', 'DM_5hpet_LPAREN_r_RPAREN_', 'EX_ser_L_LPAREN_e_RPAREN_', 'H6MTer_L', 'EX_ser_D_LPAREN_e_RPAREN_', 'EX_tacr_LPAREN_e_RPAREN_', 'DM_dem2emgacpail_prot_hs_r_']

=== au_A_aegypti ===
Mapped: 1851, Model: 1851
Missing in model: 1822, New in model: 1822
Example missing IDs: ['ALKAPHOSPHA_RXN', 'RXN_17729', 'ACID_PHOSPHATASE_RXN', 'GLUTARYL_COA_DEHYDROGENASE_RXN', 'FORMYLTETRAHYDROFOLATE_DEHYDROGENASE_RXN', 'R17_RXN', 'SAMDECARB_RXN', '4_HYDROXYPHENYLPYRUVATE_DIOXYGENASE_RXN', 'RXN66_14', 'RXN_17127']
Example new IDs: ['3.4.21.34-RXN'

In [126]:
def test_rxn_id_alignment(models_dict, mapping_df):
    results = {}

    # Asegurar que la columna "Original IDs" está como string
    mapping_df["Original ID"] = mapping_df["Original ID"].astype(str)

    for model_id, model in models_dict.items():
        # Filtrar mapping para ese organismo
        sub = mapping_df[mapping_df["model"] == model_id]
        if sub.empty:
            print(f"⚠️ No hay entradas en mapping para {model_id}")
            continue

        original_ids = set(sub["Original ID"])
        model_ids = set(model.reactions.list_attr("id"))

        total = len(original_ids)
        intersection_base = len(original_ids & model_ids)

        # Aplicar transformaciones progresivas
        def normalize_ids(ids, steps):
            out = set(ids)
            for step in steps:
                out = {step(i) for i in out}
            return out

        # Definir normalizaciones
        def to_lower(s): return s.lower()
        def to_upper(s): return s.upper()
        def underscore_to_dash(s): return s.replace("_", "-")
        def dash_to_underscore(s): return s.replace("-", "_")
        def remove_prefix_underscores(s): return re.sub(r"^_+", "", s)
        def strip_spaces(s): return s.strip()

        # Definir combinaciones a probar
        transformations = {
            "original": [],
            "lowercase": [to_lower],
            "uppercase": [to_upper],
            "underscores→dashes": [underscore_to_dash],
            "dashes→underscores": [dash_to_underscore],
            "remove leading underscores": [remove_prefix_underscores],
            "strip spaces": [strip_spaces],
            "lower+underscores→dashes": [to_lower, underscore_to_dash],
            "upper+underscores→dashes": [to_upper, underscore_to_dash],
            "lower+strip": [to_lower, strip_spaces],
            "lower+remove leading underscores": [to_lower, remove_prefix_underscores],
        }

        matches = {}

        for name, funcs in transformations.items():
            transformed = normalize_ids(original_ids, funcs)
            matches[name] = len(transformed & model_ids)

        # Crear DataFrame resumen para ese organismo
        df_summary = pd.DataFrame({
            "Transformation": matches.keys(),
            "Matched": matches.values()
        }).sort_values("Matched", ascending=False)

        results[org] = {
            "total_original": total,
            "base_match": intersection_base,
            "summary": df_summary
        }

        print(f"\n🧬 {model_id}")
        print(f" - Total IDs en mapping: {total}")
        print(f" - Coincidencias directas: {intersection_base}")
        print(df_summary)

    return results


In [127]:
# Cargar el mapping
mapping_df = pd.read_csv("mappings/complete_rxn_mapping.csv")

# Evaluar la alineación
results = test_rxn_id_alignment(models_dict, mapping_df)


🧬 ref_A_aegypti
 - Total IDs en mapping: 5678
 - Coincidencias directas: 5625
                      Transformation  Matched
0                           original     5625
5         remove leading underscores     5625
4                 dashes→underscores     5625
6                       strip spaces     5625
3                 underscores→dashes     4678
2                          uppercase     1687
8           upper+underscores→dashes     1651
1                          lowercase      740
9                        lower+strip      740
10  lower+remove leading underscores      740
7           lower+underscores→dashes      735

🧬 au_A_aegypti
 - Total IDs en mapping: 1851
 - Coincidencias directas: 29
                      Transformation  Matched
3                 underscores→dashes     1851
8           upper+underscores→dashes     1824
0                           original       29
6                       strip spaces       29
4                 dashes→underscores       29
5         remove 

In [140]:
#m = models_dict['me_A_aegypti']
#for rxn in m.reactions:
#    gpr = rxn.gene_reaction_rule
#    print(f"Reaction {rxn.id} has GPR: {gpr}")

m.reactions.get_by_id('R03673')

KeyError: 'R03673'

In [142]:
for rxn in models_dict['me_A_aegypti'].reactions:
    print(rxn.id)

R02108__extr
TO0000304_cytmem__cytmem
TI3001509_vacmem__vacmem
R04224__cytop
R00658__cytop
TO0012844_cytmem__cytmem
R01652__cytop
TI1000069_vacmem__vacmem
TO0000567_cytmem__cytmem
TI1000060_cytmem__cytmem
TO0000870_cytmem__cytmem
R00243__mito
TI3001167_cytmem__cytmem
R02596__cytop
TO1000924_cytmem__cytmem
R00021__cytop
R11143__mito
TO0000274_cytmem__cytmem
R03362__cytop
TO3000977_cytmem__cytmem
R00995__nucl
R01965__cytop
TZ3100164_cytmem__cytmem
R07889__cytop
R02731__cytop
R04922__cytop
TO0002357_cytmem__cytmem
R06728__cytop
TO1000035_cytmem__cytmem
R00102__extr
R00010__cytop
R05168__cytop
R04911__cytop
R09421__cytop
R02418__mito
R00848__mito
R02585__cytop
R11574__cytop
R08306__nucl
R08174__cytop
R09410__cytop
R10950__extr
R00623__nucl
TO0000586_vacmem__vacmem
R02503__cytop
TI3001172_cytmem__cytmem
R07084__cytop
TO0000156_cytmem__cytmem
R07109__cytop
R10630__cytop
R08183__mito
R02849__cytop
TI0000853_cytmem__cytmem
R04355__mito
R05844__cytop
R02585__extr
TI3000293_cytmem__cytmem
TO0000